# 05 — Cross-check against the human-judged tables and the narrative log

The code table (`data/processed/per_night.csv`) is the measurement. Source 1 (morning tailgate notes) and Source 2 (night
reports) are the audit trail; the narrative-log `time_lost` entries are a third, independently declared number. Nights where
all agree are high confidence. Disagreements are **listed, not averaged** (handoff §1b, §6).

Conventions: `wfrac` = weather_hours / night_hours_18; `lfrac` = nonweather_loss_hours / night_hours_18. Human P values are
reading judgements, not shares of the night.

In [ ]:
import sys
from pathlib import Path
import numpy as np, pandas as pd
sys.path.insert(0, "..")
from time_loss import DAY_OBS_WEATHER_BIT_START
pd.set_option("display.width", 220); pd.set_option("display.max_rows", 300); pd.set_option("display.max_colwidth", 90)
STEP0 = Path("../data/step0"); PROC = Path("../data/processed")

In [ ]:
t = pd.read_csv(PROC / "per_night.csv")
s1 = pd.read_csv(STEP0 / "tailgate_source1_per_night.csv"); s2 = pd.read_csv(STEP0 / "nightlog_source2_per_night.csv")
for s in (s1, s2):
    s["day_obs"] = s.day_obs.str.replace("-", "").astype(int); s["base"] = s.status.str.split().str[0]
    s["pw"] = pd.to_numeric(s.p_weather, errors="coerce"); s["pf"] = pd.to_numeric(s.p_fault, errors="coerce")
nl = pd.read_csv(STEP0 / "narrative_log_timelost_per_night.csv").rename(columns={"weather": "nl_weather_h", "fault": "nl_fault_h"})
st = pd.read_csv(STEP0 / "obs_state_per_night.csv")[["day_obs", "DOWNTIME", "FAULT"]].rename(columns={"DOWNTIME": "bit_downtime_h", "FAULT": "bit_fault_h"})
m = (t.merge(s1[["day_obs", "base", "pw", "pf", "evidence"]].add_prefix("s1_").rename(columns={"s1_day_obs": "day_obs"}), on="day_obs", how="left")
      .merge(s2[["day_obs", "base", "pw", "pf", "evidence"]].add_prefix("s2_").rename(columns={"s2_day_obs": "day_obs"}), on="day_obs", how="left")
      .merge(nl, on="day_obs", how="left").merge(st, on="day_obs", how="left"))
m[["nl_weather_h", "nl_fault_h"]] = m[["nl_weather_h", "nl_fault_h"]].fillna(0)
m["wfrac"] = m.weather_hours / m.night_hours; m["lfrac"] = m.nonweather_loss_hours / m.night_hours
m["code_closed"] = m.on_sky_any_hours < 0.5  # survey or engineering on sky (ROSP-24: on_sky_hours is survey only)
m.to_csv(PROC / "per_night_with_human.csv", index=False)
print(len(m), "nights")

## 1. CLOSED agreement

In [ ]:
print("code CLOSED (on_sky < 0.5 h) vs Source 2 status"); display(pd.crosstab(m.code_closed, m.s2_base))
print("code CLOSED vs Source 1 status"); display(pd.crosstab(m.code_closed, m.s1_base))
print("code CLOSED but a source says OPEN:", m[m.code_closed & ((m.s2_base == "OPEN") | (m.s1_base == "OPEN"))].day_obs.tolist())
print("source says CLOSED but code has >= 1 h on sky:", m[(~m.code_closed) & ((m.s2_base == "CLOSED") | (m.s1_base == "CLOSED")) & (m.on_sky_hours >= 1)][["day_obs", "on_sky_hours", "s1_base", "s2_base"]].to_string(index=False))

## 2. Weather attribution

In [ ]:
A = m[(m.wfrac >= 0.3) & (m.s2_pw <= 0.2) & (m.s2_base != "UNKNOWN")]
print(f"A. code weather >= 30 % of night, Source 2 P(weather) <= 0.2: {len(A)} nights (Dynalene is no longer weather since 2026-10-26; any remaining case is fallback cloud/C1)")
display(A[["day_obs", "weather_source", "weather_hours", "dynalene_total_hours", "on_sky_hours", "weather_and_onsky_hours", "s2_base", "s2_pw", "s2_pf"]])
B = m[(m.wfrac < 0.1) & (m.s2_pw >= 0.8) & (m.s2_base != "UNKNOWN")]
print(f"B. Source 2 P(weather) >= 0.8, code weather < 10 %: {len(B)} nights (short cloud interruptions, or bit not set)")
display(B[["day_obs", "weather_source", "weather_hours", "fallback_weather_hours", "report_cloud_hours", "on_sky_any_hours", "nonweather_loss_hours", "bit_downtime_h", "s2_base", "s2_pw"]])
print("Evidence for the B nights:")
for r in B.itertuples(): print(f"  {r.day_obs}: {str(r.s2_evidence)[:220]}")

## 3. Non-weather loss

In [ ]:
C = m[(m.lfrac >= 0.4) & (m.s2_base == "OPEN")]
print(f"C. code non-weather loss >= 40 %, Source 2 OPEN: {len(C)}"); display(C[["day_obs", "on_sky_hours", "engineering_hours", "nonweather_loss_hours", "s2_pf"]])
D = m[(m.lfrac < 0.1) & (m.s2_base.isin(["PARTIAL", "CLOSED"])) & (m.s2_pw < 0.5)]
print(f"D. Source 2 PARTIAL/CLOSED on faults, code loss < 10 %: {len(D)}"); display(D[["day_obs", "night_hours", "weather_hours", "on_sky_hours", "nonweather_loss_hours", "bit_downtime_h", "s2_base", "s2_pf"]])
for r in D.itertuples(): print(f"  {r.day_obs}: {str(r.s2_evidence)[:220]}")

## 4. Narrative-log declared time loss vs the code

In [ ]:
x = m[(m.nl_weather_h > 0) | (m.nl_fault_h > 0)]
print(f"{len(x)} nights with declared loss | corr weather {x.weather_hours.corr(x.nl_weather_h):.2f} | corr fault {x.nonweather_loss_hours.corr(x.nl_fault_h):.2f}")
print(f"sums: code weather {x.weather_hours.sum():.0f} h vs declared weather {x.nl_weather_h.sum():.0f} h; code non-weather {x.nonweather_loss_hours.sum():.0f} h vs declared fault {x.nl_fault_h.sum():.0f} h")
y = x[x.nl_fault_h < 12]
print(f"without the whole-night camera-shutdown tallies (declared fault >= 12 h): code non-weather {y.nonweather_loss_hours.sum():.0f} h vs declared fault {y.nl_fault_h.sum():.0f} h, corr {y.nonweather_loss_hours.corr(y.nl_fault_h):.2f}")
import matplotlib.pyplot as plt
from time_loss.style import COLORS
fig, axs = plt.subplots(1, 2, figsize=(9, 4))
axs[0].scatter(y.nl_fault_h, y.nonweather_loss_hours, s=12, color=COLORS["nonweather_loss"]); axs[0].plot([0, 10], [0, 10], color=COLORS["grid"]); axs[0].set_xlabel("narrative log: declared fault [h]"); axs[0].set_ylabel("code: non-weather loss [h]")
axs[1].scatter(x.nl_weather_h, x.weather_hours, s=12, color=COLORS["weather"]); axs[1].plot([0, 10], [0, 10], color=COLORS["grid"]); axs[1].set_xlabel("narrative log: declared weather [h]"); axs[1].set_ylabel("code: weather [h]")
fig.tight_layout()

## 5. Confidence per night

HIGH: code label consistent with both sources (or the one that exists). MEDIUM: consistent with one source. LOW: contradicts both, or both UNKNOWN.

In [ ]:
def consistent(row, base, pw):
    if pd.isna(base) or base == "UNKNOWN": return np.nan
    if row.label == "CLOSED": return base == "CLOSED" or (base == "PARTIAL")  # PARTIAL (≈ closed) rows exist
    if row.label == "WEATHER-DOMINATED": return (pw >= 0.5) and base != "CLOSED"
    if row.label == "FAULT-DOMINATED": return base in ("PARTIAL", "OPEN") and (pw < 0.8)
    return base == "OPEN"  # OPEN
m["ok_s1"] = [consistent(r, r.s1_base, r.s1_pw) for r in m.itertuples()]
m["ok_s2"] = [consistent(r, r.s2_base, r.s2_pw) for r in m.itertuples()]
def conf(r):
    v = [x for x in (r.ok_s1, r.ok_s2) if not pd.isna(x)]
    if not v: return "LOW (no human source)"
    if all(v): return "HIGH" if len(v) == 2 else "MEDIUM (one source)"
    return "MEDIUM (one source)" if any(v) else "LOW (contradicts)"
m["confidence"] = m.apply(conf, axis=1)
m.to_csv(PROC / "per_night_with_human.csv", index=False)
print(m.confidence.value_counts())
m[m.confidence.str.startswith("LOW")][["day_obs", "label", "weather_hours", "on_sky_hours", "nonweather_loss_hours", "s1_base", "s1_pw", "s2_base", "s2_pw"]]